<a href="https://colab.research.google.com/github/VanshChauhan-0001/Ml_Models/blob/main/Review_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install Kaggle

In [2]:
import os
import json

from zipfile import ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
Kaggle_dictionary = json.load(open('kaggle.json'))
os.environ['KAGGLE_USERNAME'] = Kaggle_dictionary['username']
os.environ['KAGGLE_KEY'] = Kaggle_dictionary['key']

In [4]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
 39% 10.0M/25.7M [00:00<00:00, 104MB/s]
100% 25.7M/25.7M [00:00<00:00, 153MB/s]


In [5]:
!ls

imdb-dataset-of-50k-movie-reviews.zip  kaggle.json  sample_data


In [6]:
with ZipFile('imdb-dataset-of-50k-movie-reviews.zip', 'r') as zip_ref:
  zip_ref.extractall()

In [7]:
!ls

'IMDB Dataset.csv'   imdb-dataset-of-50k-movie-reviews.zip   kaggle.json   sample_data


In [8]:
data = pd.read_csv('IMDB Dataset.csv')

In [14]:
data.replace({'sentiment' : {"positive":1,"negative":0}}, inplace = True)

<ipython-input-14-66873a5ada0e>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({'sentiment' : {"positive":1,"negative":0}}, inplace = True)


In [15]:
data['sentiment'].value_counts()

,count
sentiment,
1,25000
0,25000


In [16]:
train_data, test_data = train_test_split(data, test_size = 0.2, random_state = 42)

In [19]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data['review'])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']),maxlen = 200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']),maxlen = 200)

In [20]:
print(X_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [21]:
print(X_test)

[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [31]:
model = Sequential()
model.add(Embedding(input_dim = 5000, output_dim = 128, input_length = 200))
model.add(LSTM(128, dropout = 0.2, recurrent_dropout = 0.2))
model.add(Dense(1, activation = 'sigmoid'))

In [33]:
model.build((1, 200))  # Replace 1 with your desired batch size

In [34]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)              │ (1, 200, 128)               │         640,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_4 (LSTM)                        │ (1, 128)                    │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (1, 1)                      │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [36]:
model.fit(X_train, train_data['sentiment'], epochs = 3, batch_size = 64)

Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 295s 468ms/step - accuracy: 0.7331 - loss: 0.5161
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 321s 467ms/step - accuracy: 0.8182 - loss: 0.4084
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 305s 441ms/step - accuracy: 0.8438 - loss: 0.3653


In [37]:
loss, accuracy = model.evaluate(X_test, test_data['sentiment'])
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 109ms/step - accuracy: 0.8644 - loss: 0.3195
Test Loss: 0.32052817940711975, Test Accuracy: 0.8658999800682068


In [38]:
def predict_sentiment(review):
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen = 200)
  prediction = model.predict(padded_sequence)
  sentiment = 'positive' if prediction > 0.5 else 'negative'
  return sentiment

In [41]:
new_review = "This movie was fantastic! I loved every minute of it."
predicted_sentiment = predict_sentiment(new_review)
print(f'Predicted Sentiment: {predicted_sentiment}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicted Sentiment: positive


In [42]:
new_review = "The acting was terrible and the plot was boring."
predicted_sentiment = predict_sentiment(new_review)
print(f'Predicted Sentiment: {predicted_sentiment}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Predicted Sentiment: negative
